[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1BoazozMT52he6rRSPq14zbG8UzwFG_cj/view?usp=drive_link)

# Workflow Evaluation

This notebook demonstrates how to evaluate multi-agent workflows where multiple agents work together in a DAG (directed acyclic graph). Floeval runs the workflow for each test case, captures traces from each agent, and scores the overall workflow output.

**FloTorch Console:** [https://docs.flotorch.cloud/introduction/](https://docs.flotorch.cloud/introduction/)

**Prerequisites**

1. **Configure agents in the FloTorch Console** — Create and deploy your agents in the [FloTorch Console](https://docs.flotorch.cloud/introduction/). Each agent must be deployed before you can reference it in the DAG. See [Agent Builder](https://docs.flotorch.cloud/gateway/agents/) and [Workflows](https://docs.flotorch.cloud/gateway/agents/workflows/) for details.

2. **Create an API key** — Create an API key in [Settings > API Keys](https://docs.flotorch.cloud/workspace/settings/apikeys/) for authentication.

3. **Install FloTorch integration** — Run `pip install floeval[flotorch]`.

**Objectives**
- Define a DAG config with nodes (START, AGENT, END) and edges
- Create a `WorkflowRunner` from the DAG config
- Run `AgentEvaluation` with the workflow runner and inspect results

## 1. Installation

Install Floeval with FloTorch support. Required for evaluating multi-agent workflows on the FloTorch gateway.

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

In [ ]:
%pip install floeval[flotorch]

## 2. Configuration Constants

Set the following constants before running. Obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/). Update the DAG config with your deployed agent names.

**Provider flexibility:** Workflow evaluation uses the FloTorch gateway. To use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# FloTorch gateway configuration
FLOTORCH_BASE_URL = "https://gateway.flotorch.cloud/openai/v1"
FLOTORCH_API_KEY = getpass.getpass("your-gateway-key")
FLOTORCH_CHAT_MODEL = "gpt-4o-mini"
FLOTORCH_EMBEDDING_MODEL = "text-embedding-3-small"
AGENT1 = "agent-1" # Agents configured in flotorch console.
AGENT2 = "agent-2"

## 3. Imports

The following cell imports the agent evaluation components, the dataset schemas, `WorkflowRunner`, and the LLM configuration schema.

In [ ]:
from floeval.api.agent_evaluation import AgentEvaluation
from floeval.config.schemas.io.agent_dataset import AgentDataset, PartialAgentSample
from floeval.config.schemas.io.llm import OpenAIProviderConfig
from floeval.flotorch import WorkflowRunner

## 4. Configure the LLM

The LLM configuration is built for the FloTorch gateway. Set `FLOTORCH_BASE_URL` and `FLOTORCH_API_KEY` in your environment or pass them explicitly. Credentials are available from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=FLOTORCH_BASE_URL,
    api_key=FLOTORCH_API_KEY,
    chat_model=FLOTORCH_CHAT_MODEL,
    embedding_model=FLOTORCH_EMBEDDING_MODEL,
)

## 5. Define the DAG Config

The DAG config specifies the workflow structure. Each `AGENT` node has:
- **id**: Unique node identifier (used in edges)
- **callableName**: Deployed agent name on the FloTorch gateway — must match agents in the [FloTorch Console](https://docs.flotorch.cloud/introduction/)

This example defines a sequential workflow: agent1 (get-topics) runs first, then agent2 (content-writer).

In [ ]:
dag_config = {
    "uid": "sequential-workflow-001",
    "name": "Sequential Workflow",
    "nodes": [
        {"id": "start", "type": "START", "label": "Start"},
        {"id": "agent1", "type": "AGENT", "label": "Agent 1", "callableName": AGENT1},
        {"id": "agent2", "type": "AGENT", "label": "Agent 2", "callableName": AGENT2},
        {"id": "end", "type": "END", "label": "End"},
    ],
    "edges": [
        {"sourceNodeId": "start", "targetNodeId": "agent1"},
        {"sourceNodeId": "agent1", "targetNodeId": "agent2"},
        {"sourceNodeId": "agent2", "targetNodeId": "end"},
    ],
}
# Note: callableName = deployed agent name on FloTorch gateway
print("DAG config defined")

## 6. Create the Workflow Runner

The `WorkflowRunner` is instantiated with the DAG config and LLM config. It executes the workflow by calling each agent node according to the DAG edges.

In [ ]:
runner = WorkflowRunner(dag_config=dag_config, llm_config=llm_config)
print("WorkflowRunner created")

## 7. Prepare the Dataset

Each sample represents a test case for the full workflow. Samples include `user_input` and `reference_outcome`. The workflow runner sends the input through all agent nodes in the DAG according to the defined edges.

In [ ]:
dataset = AgentDataset(
    samples=[
        PartialAgentSample(
            user_input="My order has not arrived after two weeks.",
            reference_outcome="An apology and a case escalation to the shipping team.",
        ),
        PartialAgentSample(
            user_input="What is the status of order #12345?",
            reference_outcome="The order is shipped and arriving tomorrow.",
        ),
    ]
)
print(f"Dataset loaded: {len(dataset.samples)} sample(s)")

## 8. Run the Workflow and Create Full Samples

`WorkflowRunner.run_on_dataset` is async. Run it first to get full samples with traces, then pass the full dataset to `AgentEvaluation` for scoring. This avoids the async/sync mismatch when using `agent_runner` with workflow evaluation.

In [ ]:
# WorkflowRunner.run_on_dataset is async — run it to get full samples with traces.
full_samples = await runner.run_on_dataset(dataset.all_partial)
full_dataset = AgentDataset(samples=full_samples)

evaluation = AgentEvaluation(
    dataset=full_dataset,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence", "ragas:agent_goal_accuracy"],
    default_provider="builtin",
)

results = evaluation.run()

# Enrich sample_results with agent_traces from full samples (per user guide)
for i, row in enumerate(results.sample_results):
    if i < len(full_samples):
        s = full_samples[i]
        row["agent_traces"] = s.agent_traces or []
        row["workflow_execution"] = (s.metadata or {}).get("workflow_execution")

print("Summary:", results.summary)

## 9. Inspect Per-Sample Results

Each sample result includes `final_response` and metric scores. This enables per-sample analysis of workflow evaluation quality.

In [ ]:
# Per-sample: agent_traces (one per DAG node) and workflow_execution dict
for row in results.sample_results:
    print("Final response:", row.get("final_response"))
    print("Agent traces:", len(row.get("agent_traces", [])), "nodes")
    for k, v in row.get("metrics", {}).items():
        print(f"  {k}: score={v.get('score')}")

## Summary

This notebook demonstrated the evaluation of multi-agent workflows using a DAG structure.

The key components included:

1. **LLM Configuration**: The FloTorch gateway credentials were configured for the evaluation.
2. **DAG Definition**: A DAG config with START, AGENT, and END nodes was defined. Agents must be configured in the [FloTorch Console](https://docs.flotorch.cloud/introduction/) before use.
3. **Workflow Runner**: A `WorkflowRunner` was created from the DAG config and LLM config.
4. **Dataset Preparation**: A partial agent dataset was built with workflow test cases.
5. **Workflow Execution**: `runner.run_on_dataset()` was called to run the workflow and capture traces.
6. **Evaluation Execution**: The full dataset (with traces) was passed to `AgentEvaluation` for scoring.
7. **Metrics**: The `goal_achievement`, `response_coherence`, and `ragas:agent_goal_accuracy` metrics were run.
8. **Results Inspection**: The summary and per-sample results were accessed through the results object.

This example showcases the workflow for evaluating multi-agent DAG workflows on the FloTorch gateway.